# A5a · GLM-5.3 (MLA + DSA indexer 공유 + MoE) 검증

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_llm_dit/blob/main/notebooks/verify/A5a_glm53.ipynb)

원본: transformers `glm_moe_dsa` (GLM-5 ~ 5.3 공통, 순수 PyTorch). 내 구현: `miniarch/models/glm53.py`.

- MLA: q 저랭크(q_a → norm → q_b), KV 를 작은 잠재 c 로 압축한 뒤 head 별 k_nope / v 로 펼침, RoPE 는 공유 k_rope 채널에만 (interleaved)
- DSA: Lightning Indexer 가 질의마다 이전 토큰 중 top-k 만 고르고 attention 은 그 토큰만 봄.
  `indexer_types` 가 "shared" 인 층은 indexer 없이 직전 "full" 층의 선택을 재사용 (원본: 앞 3층 full, 이후 4층마다 full)
- MoE: sigmoid 점수 + 그룹 제한 top-k + aux-loss-free bias(선택에만) + shared expert, 앞쪽 층은 dense MLP

축소 config: 4층, indexer = [full, shared, shared, full], MLP = [dense, sparse, sparse, sparse],
indexer top-6 에 시퀀스 24 → 앞쪽 질의(이전 토큰 6개 이하)는 전부 선택, 뒤쪽은 일부만 선택되는 두 경로가 모두 쓰입니다.
shared 가 두 번 이어진 뒤 다시 full 이 나와 선택이 넘어가고 새로 계산되는 경로를 봅니다.
expert 8개를 4그룹으로 나누고 2그룹만 허용, top-3 → 그룹 마스킹 경로를 씁니다 (원본은 n_group=1).
bias 경로도 검사하려고 attention_bias=True 로 둡니다 (원본은 False). 라우팅 bias 버퍼는 0이 아닌 값으로 채웁니다.

순서: 0. 환경 준비 → 1. 원본 구조 (HF 헤더) → 2. 원본 모델 동작 확인 → 3. 내 구현 == 원본 (층별 출력 비교) → 4. causal 누수 → 5. 학습 동작 확인 → 6. 단계 전용 자체 점검 → 7. 결과 저장

**런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 위에서부터 실행하세요. 다른 모델 노트북과 독립적으로 돌아갑니다.

## 0. 환경 준비

In [ ]:
import os
REPO_DIR = "/content/tmp_llm_dit"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/HisameOgasahara/tmp_llm_dit.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
if not os.environ.get("MINIARCH_SKIP_PIP"):   # 전체 실행 노트북이 한 번만 설치하고 넘길 때 건너뜀
    !pip -q install -U -r requirements-colab.txt
    # Colab 기본 torchao 가 최신 diffusers 와 버전이 안 맞아 import 가 깨짐 → 쓰지 않으므로 제거
    !pip -q uninstall -y torchao

In [ ]:
import sys, torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (런타임 유형을 GPU로 바꾸세요)")
!git log -1 --format='repo commit: %h %s'

In [ ]:
from miniarch import verify as V
from miniarch.verify import stages
S = stages.get("A5a")
dev = "cuda" if torch.cuda.is_available() else "cpu"
try:   # gated 모델 헤더/원격 코드용. Colab 왼쪽 🔑(보안 비밀)에 HF_TOKEN 등록.
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

CHECKS = []   # (이름, 통과, 오차) → 마지막에 저장
def report(name, ok, err=float("nan")):
    CHECKS.append({"check": name, "ok": bool(ok), "max_abs": float(err)})
    print(f"{'✅' if ok else '❌'} {name:<44} max_abs={err:.2e}")

## 원본 구조 (HF 헤더)

실제 체크포인트의 safetensors 헤더만 읽어 가중치 이름, shape, dtype을 봅니다 (다운로드 없음).
**구현 검증이 아니라 원본 구조 확인용**입니다. routed expert는 개수가 많아 숨깁니다.

In [ ]:
def peek(repo, pattern, subfolder="", limit=60):
    print(f"\n===== {repo}  /{pattern}/")
    try:
        cfg = V.fetch_config(repo, subfolder)
        cfg = cfg.get("text_config", cfg)
        keys = ["model_type", "hidden_size", "num_hidden_layers", "num_attention_heads", "num_key_value_heads", "head_dim"]
        print({k: cfg[k] for k in keys if k in cfg})
        h = V.read_model_header(repo, subfolder=subfolder, pattern=pattern)
        V.show({k: v for k, v in h.items() if ".experts." not in k}, limit)
    except Exception as e:
        print(f"⚠️ {type(e).__name__}: {e}")

peek('zai-org/GLM-5.3-BF16', 'layers\\.(2|3)\\.', subfolder='')

## 원본 모델 동작 확인

축소 config로 원본을 만들고 한 번 forward 합니다. 여기서 실패하면 라이브러리 버전 문제입니다 (내 구현과 무관).

In [ ]:
V.set_deterministic(0)
ref, inputs = S.ref()
ref = ref.to(dev).eval()
with torch.no_grad():
    out = ref(**{k: v.to(dev) if torch.is_tensor(v) else v for k, v in inputs.items()})
t = out[0] if isinstance(out, tuple) else getattr(out, "last_hidden_state", None)
t = t if t is not None else out.sample
print(f"{type(ref).__name__}  params={sum(p.numel() for p in ref.parameters()):,}  out={tuple(t.shape)}")

## 내 구현 == 원본

원본 가중치에 잡음을 넣고(초기값 0/1 때문에 복사 누락이 가려지지 않도록) 내 구현에 복사한 뒤, 층별 출력과 최종 출력을 비교합니다.

In [ ]:
row = V.cases.run_case(S.CASE)
print(row["status"]); print(row["detail"])
report("원본 동치 (" + S.CASE.reference + ")", row["status"].startswith("✅"))

## causal 누수

뒤쪽 토큰을 바꿨을 때 앞쪽 출력이 변하면 미래 정보가 새고 있다는 뜻입니다.

In [ ]:
import importlib
V.set_deterministic(0)
mine = importlib.import_module(S.CASE.mine_module).verify_pair(S.ref()[0])["mine"].to(dev).eval()
r = V.causal_leak_test(lambda x: mine(x).last_hidden_state, vocab_size=V.cases.VOCAB, seq_len=48, device=dev, atol=1e-4)
report("내 구현 " + r.name, r.ok, r.max_abs)

## 학습 동작 확인

forward 가 원본과 같아도 학습이 되는지는 따로 확인해야 합니다.
- **기울기 == 원본**: 같은 가중치·같은 손실로 원본과 내 구현을 backward 해서 파라미터 기울기를 비교합니다 (파라미터별 상대 오차 ≤ 1e-3).
  원본이 추론 전용 코드(no_grad)면 건너뜁니다.
- **학습 궤적 == 원본**: 같은 가중치·데이터·optimizer(SGD momentum 0.9, 30 step)로 원본과 내 구현을 나란히 학습해
  step 별 손실과 최종 가중치를 비교합니다 (상대차 ≤ 1e-3). Adam 은 아주 작은 잡음 차이를 키워서 궤적 비교에 쓰지 않습니다.
- **배치 하나 외우기**: 고정 배치 하나로 300 step 학습해 손실이 처음의 20% 이하로 내려가는지 봅니다 (LM은 다음 토큰 CE, 그 외는 고정 목표 MSE).
  NaN/inf 가 없어야 하고, 기울기를 한 번도 받지 못한 파라미터를 보여줍니다 (MoE 에서 안 뽑힌 expert 는 정상).

In [ ]:
import importlib, traceback
from miniarch.verify import training as TR
mod = importlib.import_module(S.CASE.mine_module)
V.set_deterministic(0)
ref, inputs = S.ref()
V.perturb_params(ref)
pair = mod.verify_pair(ref)
try:
    g = TR.grad_equivalence(ref, pair, inputs, dev)
    if g.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", g.note); report("기울기 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"비교한 파라미터 {g.compared}개, 최대 상대 오차 {g.max_rel:.2e} ({g.worst})", g.note)
        if g.skipped_ref:
            print("원본에서 기울기 없는 파라미터:", g.skipped_ref[:8])
        report("기울기 == 원본", g.ok, g.max_rel)
except Exception:
    traceback.print_exc(limit=4); report("기울기 == 원본", False)

In [ ]:
try:
    t = TR.train_parity(ref, pair, inputs, dev)
    if t.note.startswith("원본 출력에 기울기가 없음"):
        print("⏭️", t.note); report("학습 궤적 == 원본 (원본 추론 전용 → 건너뜀)", True)
    else:
        print(f"{t.steps} step  손실 최대 상대차 {t.max_loss_rel:.2e}  최종 가중치 최대 상대차 {t.max_param_rel:.2e} ({t.worst_param})", t.note)
        print("  원본:", " ".join(f"{v:.4g}" for v in t.loss_ref[::6]))
        print("  내것:", " ".join(f"{v:.4g}" for v in t.loss_mine[::6]))
        report("학습 궤적 == 원본 (SGD 30 step)", t.ok, max(t.max_loss_rel, t.max_param_rel))
except Exception:
    traceback.print_exc(limit=4); report("학습 궤적 == 원본", False)

In [ ]:
try:
    o = TR.overfit_batch(pair["mine"], pair, inputs, dev)
    print(f"{o.kind}: {o.loss_start:.4f} → {o.loss_end:.4f}  (유한={o.finite})")
    print("  손실:", " ".join(f"{v:.3g}" for v in o.curve[::30]))
    if o.no_grad_params:
        print(f"  기울기를 못 받은 파라미터 {len(o.no_grad_params)}개:", o.no_grad_params[:8])
    report("배치 하나 외우기 (" + o.kind + ")", o.ok, o.loss_end / max(o.loss_start, 1e-12))
except Exception:
    traceback.print_exc(limit=4); report("배치 하나 외우기", False)

## 단계 전용 자체 점검

### attention map · top-k 선택 · 층 간 공유 확인

- `attn_probs`: [B, H, T, T]. 행 합 = 1, 선택되지 않은 칸의 확률은 정확히 0이어야 합니다.
- `sel_probe`: 실제로 보는 칸 (causal ∧ top-k). 질의마다 최대 index_topk 개.
- "shared" 층의 선택은 직전 "full" 층과 완전히 같아야 합니다.

In [ ]:
from miniarch.models import glm53 as G
ref, _ = S.ref()
m = G.verify_pair(ref)["mine"].to(dev).eval()
names = [n for n, _ in m.named_modules() if n.endswith(("attn_probs", "sel_probe"))]
x = torch.randint(0, V.cases.VOCAB, (2, 24), device=dev)
with V.capture(m, names) as st, torch.no_grad():
    m(x)
c, last = m.config, None
for i in range(c.n_layer):
    p, sel = st[f"layers.{i}.self_attn.attn_probs"], st[f"layers.{i}.self_attn.sel_probe"]
    err = (p.sum(-1) - 1).abs().max().item()
    leak = p.masked_fill(sel[:, None], 0).abs().max().item()
    n_sel = int(sel.sum(-1).max().item())
    ok = err < 1e-5 and leak == 0 and n_sel <= c.index_topk
    if c.indexer_types[i] == "full":
        last = sel
    else:
        ok = ok and torch.equal(sel, last)
    print(f"layer {i} {c.indexer_types[i]:<6} probs={tuple(p.shape)} 행합 오차={err:.1e} 비선택 확률={leak:.1e} 최대 선택={n_sel}")
    report(f"layer {i} ({c.indexer_types[i]}) top-k 마스크", ok, err)

### top-k ≥ 시퀀스 길이면 DSA == dense causal attention

indexer 가 모든 이전 토큰을 고르면 sparse attention 은 일반 causal attention 과 같아야 합니다
(`use_dsa=False` 스위치와 같은 가중치로 비교).

In [ ]:
import dataclasses
from miniarch.models import glm53 as G
V.set_deterministic(0)
c = dataclasses.replace(G.config_from_hf(S.ref()[0].config), index_topk=10**6)
m1 = G.GLM53(c).to(dev).eval()
m2 = G.GLM53(dataclasses.replace(c, use_dsa=False)).to(dev).eval()
m2.load_state_dict(m1.state_dict())
x = torch.randint(0, V.cases.VOCAB, (2, 24), device=dev)
r = V.same_function_test("DSA(top-k ≥ T) == dense causal", lambda: m1(x).last_hidden_state,
                         lambda: m2(x).last_hidden_state, atol=1e-5)
report(r.name, r.ok, r.max_abs)

## 결과 저장

- `results/`에 점검 결과를 저장합니다.
- 노트북 출력까지 남기려면 **파일 → GitHub에 사본 저장** → 이 저장소의 `notebooks/verify/A5a_glm53.ipynb`.

In [ ]:
import datetime, pandas as pd
os.makedirs("results", exist_ok=True)
df = pd.DataFrame(CHECKS)
p = f"results/verify_A5a_{datetime.datetime.now():%Y%m%d-%H%M}.csv"
df.to_csv(p, index=False)
print(df.to_string(index=False)); print("saved:", p)
print("전체:", "✅ 모두 통과" if df.ok.all() else "❌ 실패 있음")